# RT-DETR-L matched a YOLOv8m-seg V4

**Objetivo:** entrenar RT-DETR-L con el **mismo régimen de entrenamiento** que YOLOv8m-seg V4 para que la comparación sea justa.

**Variables igualadas (las que afectan al aprendizaje):**
- `epochs=120`, `patience=25`
- `batch=8`, `imgsz=640`
- `seed=42`, `cos_lr=True`, `warmup_epochs=3`, `close_mosaic=20`
- Augmentations idénticas (hsv, degrees, scale, shear, mosaic, mixup, copy_paste, erasing, etc.)
- Mismo dataset y split (`BlackjackVAI-4`, train/val)

**Variables específicas por arquitectura (no se igualan, cada uno con su óptimo):**
- Optimizer: RT-DETR usa AdamW (default), YOLO usa SGD (auto)
- `lr0=0.0001` para RT-DETR (AdamW) vs `lr0=0.001` para YOLO V4 (SGD, fine-tune desde V3)
- Modelo base: RT-DETR parte de `rtdetr-l.pt` (COCO), YOLO V4 partió de su V3 (transfer interno)

**Tracking:** experimento MLflow `blackjackvai-rtdetr-matched` en la misma `mlflow.db` consolidada.

## Setup Colab (no-op en local)

Detecta si estás en Colab. Si lo estás:
- Instala `ultralytics`, `mlflow`, `roboflow`
- Verifica GPU (T4 / V100 / A100)
- Crea estructura de directorios

Si estás en local con todo instalado, esta celda no rompe nada.

In [ ]:
import sys, subprocess, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print(f"Entorno: {'Colab' if IN_COLAB else 'local'}")

if IN_COLAB:
    print("Instalando dependencias...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "ultralytics", "mlflow", "roboflow", "python-dotenv", "pyyaml"])
    print("Dependencias OK")
    # Working dir en Colab: /content
    os.chdir("/content")
    Path("/content/blackjack-VAI").mkdir(exist_ok=True)
    os.chdir("/content/blackjack-VAI")
    print(f"CWD: {os.getcwd()}")

import torch
if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print(f"GPU: {g.name}  |  VRAM: {g.total_memory/1e9:.1f} GB")
    if g.total_memory/1e9 < 14:
        print("AVISO: <14 GB VRAM — si OOM con batch=8, baja a batch=4")
else:
    print("AVISO: sin GPU — Runtime > Cambiar tipo de entorno > GPU")

## 0. Configuración

In [ ]:
import os
from pathlib import Path

BASE_DIR    = Path(".").resolve()
DATASET_DIR = BASE_DIR / "BlackjackVAI-4"
DATA_YAML   = DATASET_DIR / "data.yaml"
WORKDIR     = BASE_DIR / "training_runs"
MODELS_DIR  = BASE_DIR / "models"

RUN_NAME    = "rtdetr_l_matched"
MODEL_BASE  = "rtdetr-l.pt"   # COCO pretrained, ultralytics lo descarga si falta
EPOCHS      = 120             # igualado a YOLO V4
IMGSZ       = 640
BATCH       = 8              # fallback a 4 si OOM (gestionado en celda de entreno)
PATIENCE    = 25             # igualado a YOLO V4
SEED        = 42
SAVE_PERIOD = 10
RESUME      = True

# MLflow — misma DB consolidada que el resto de runs
MLFLOW_EXPERIMENT = "blackjackvai-rtdetr-matched"
MLFLOW_DB         = f"sqlite:///{(BASE_DIR / 'mlflow.db').as_posix()}"
os.environ["MLFLOW_TRACKING_URI"]    = MLFLOW_DB
os.environ["MLFLOW_EXPERIMENT_NAME"] = MLFLOW_EXPERIMENT

RUN_DIR = WORKDIR / "runs" / RUN_NAME
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"

WORKDIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset           : {DATASET_DIR}  {'(existe)' if DATA_YAML.exists() else '(se descargará en la celda 1b)'}")
print(f"Modelo base       : {MODEL_BASE}  (COCO pretrained)")
print(f"Run dir           : {RUN_DIR}")
print(f"MLflow DB         : {MLFLOW_DB}")
print(f"MLflow experiment : {MLFLOW_EXPERIMENT}")
print(f"Resume            : {RESUME}")

## 1b. Descarga del dataset desde Roboflow + fusión test→train

Solo descarga si `BlackjackVAI-4/` no existe. Después fusiona `test/` en `train/` para replicar exactamente el split 80/20 que usó YOLO V4.

**Credenciales:**
- En Colab: pon tu `ROBOFLOW_API_KEY` en *Secrets* (icono de la llave a la izquierda) o introdúcela cuando pida.
- En local: lee de `.env` si existe, o del entorno.

In [ ]:
import shutil

# Re-resolver BASE_DIR por si chdir en Colab cambió el cwd
BASE_DIR    = Path(".").resolve()
DATASET_DIR = BASE_DIR / "BlackjackVAI-4"
DATA_YAML   = DATASET_DIR / "data.yaml"

ROBOFLOW_WORKSPACE = "javiers-workspace-q8mnr"
ROBOFLOW_PROJECT   = "blackjackvai"
ROBOFLOW_VERSION   = 4

def get_roboflow_key():
    # 1) Colab secrets
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("ROBOFLOW_API_KEY")
            if k: return k
        except Exception:
            pass
    # 2) Env / .env
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    k = os.environ.get("ROBOFLOW_API_KEY")
    if k: return k
    # 3) Input interactivo
    from getpass import getpass
    return getpass("ROBOFLOW_API_KEY: ").strip()

if DATA_YAML.exists():
    print(f"Dataset ya existe en {DATASET_DIR} — saltando descarga.")
else:
    print(f"Descargando dataset V{ROBOFLOW_VERSION} desde Roboflow...")
    from roboflow import Roboflow
    rf = Roboflow(api_key=get_roboflow_key())
    project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)
    dataset = project.version(ROBOFLOW_VERSION).download("yolov8", location=str(DATASET_DIR))
    print(f"Dataset descargado en: {dataset.location}")

# Fusión test → train (replicando split 80/20 de V4)
test_img_dir  = DATASET_DIR / "test" / "images"
test_lbl_dir  = DATASET_DIR / "test" / "labels"
train_img_dir = DATASET_DIR / "train" / "images"
train_lbl_dir = DATASET_DIR / "train" / "labels"

if test_img_dir.exists():
    merged = 0
    for img in test_img_dir.glob("*.*"):
        dest = train_img_dir / img.name
        if not dest.exists():
            shutil.copy2(img, dest); merged += 1
    for lbl in test_lbl_dir.glob("*.txt"):
        dest = train_lbl_dir / lbl.name
        if not dest.exists():
            shutil.copy2(lbl, dest)
    print(f"Fusionadas {merged} imágenes de test → train")
else:
    print("Carpeta test/ no encontrada — split ya es 80/20")

for split in ["train", "val", "valid", "test"]:
    d = DATASET_DIR / split / "images"
    if d.exists():
        n = len(list(d.glob("*.*")))
        print(f"  {split}: {n} imágenes")

## 1. Dependencias y verificación de GPU

In [ ]:
import torch

DEVICE = 0 if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    vram_gb = gpu.total_memory / 1e9
    print(f"GPU : {gpu.name}  |  VRAM: {vram_gb:.1f} GB")
    if vram_gb < 8:
        print("AVISO: < 8 GB VRAM — considera BATCH=4 (RT-DETR-L es pesado)")
else:
    print("AVISO: sin GPU — el entreno será inviable en CPU")

import ultralytics, mlflow
print(f"Ultralytics {ultralytics.__version__} | MLflow {mlflow.__version__} | PyTorch {torch.__version__}")

## 2. Verificar data.yaml y limpiar caches

Asume que `BlackjackVAI-4/` ya está en el repo con el split 80/20 (train/val) tras la fusión de test→train hecha en V4. RT-DETR usa el bbox que encierra los polígonos de segmentación, ignorando las máscaras.

In [ ]:
import yaml

with open(DATA_YAML, encoding="utf-8") as f:
    data = yaml.safe_load(f)

data["path"] = str(DATASET_DIR.resolve())
# Normaliza 'valid' → 'val' si hace falta
if "valid" in data and "val" not in data:
    data["val"] = data.pop("valid")
data.pop("test", None)

with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

names = data["names"] if isinstance(data["names"], list) else list(data["names"].values())
print(f"Clases ({data['nc']}): {names[:8]} ...")
print(f"Train : {data.get('train')}")
print(f"Val   : {data.get('val')}")

# Limpia caches viejas
for cache in DATASET_DIR.rglob("*.cache"):
    cache.unlink()
    print(f"Cache eliminada: {cache}")

## 3. Configurar MLflow

In [ ]:
from ultralytics import settings as yolo_settings
yolo_settings.update({"mlflow": True})

print(f"YOLO mlflow activo  : {yolo_settings.get('mlflow')}")
print(f"MLflow URI          : {os.environ['MLFLOW_TRACKING_URI']}")
print(f"MLflow experiment   : {os.environ['MLFLOW_EXPERIMENT_NAME']}")

## 4. Entrenamiento

### Filosofía de matching

| Tipo | Variable | Valor | Justificación |
|------|----------|-------|---------------|
| **Igualada** | epochs | 120 | mismo budget que YOLO V4 |
| **Igualada** | patience | 25 | mismo criterio de early stop |
| **Igualada** | batch | 8 | mismo tamaño efectivo |
| **Igualada** | imgsz | 640 | misma resolución |
| **Igualada** | seed | 42 | misma reproducibilidad |
| **Igualada** | warmup_epochs | 3 | mismo warmup que V4 |
| **Igualada** | augmentations | idénticas | mismo régimen de regularización |
| **Específica** | optimizer | AdamW (auto) | óptimo para RT-DETR (paper) |
| **Específica** | lr0 | 0.0001 | óptimo para AdamW |

### Fallback OOM

Si la primera iteración revienta VRAM, baja `BATCH` a 4 en la celda 0 y vuelve a ejecutar. RT-DETR-L con `imgsz=640` y `batch=8` ocupa ~7–7.5 GB en AMP — va justo en RTX 4070.

In [ ]:
from ultralytics import RTDETR

resuming = RESUME and LAST_PT.exists()
if resuming:
    print(f"Reanudando desde: {LAST_PT}")
    model = RTDETR(str(LAST_PT))
else:
    print(f"Entrenando desde cero: {MODEL_BASE}")
    model = RTDETR(MODEL_BASE)

train_kwargs = dict(
    data         = str(DATA_YAML),
    epochs       = EPOCHS,
    imgsz        = IMGSZ,
    batch        = BATCH,
    device       = DEVICE,
    workers      = 4,
    seed         = SEED,
    pretrained   = True,
    amp          = True,
    cache        = False,
    cos_lr       = True,
    lr0          = 0.0001,       # ÓPTIMO RT-DETR (AdamW)
    lrf          = 0.01,
    warmup_epochs= 3,            # igualado a YOLO V4
    momentum     = 0.937,
    weight_decay = 0.0005,
    patience     = PATIENCE,
    save_period  = SAVE_PERIOD,
    plots        = True,
    save         = True,
    project      = str(WORKDIR / "runs"),
    name         = RUN_NAME,
    exist_ok     = True,
    rect         = False,
    # ── Augmentations idénticas a YOLO V4 ──
    hsv_h        = 0.02,
    hsv_s        = 0.90,
    hsv_v        = 0.60,
    degrees      = 20.0,
    translate    = 0.20,
    scale        = 0.85,
    shear        = 8.0,
    perspective  = 0.001,
    flipud       = 0.25,
    fliplr       = 0.50,
    mosaic       = 1.0,
    mixup        = 0.25,
    copy_paste   = 0.60,
    copy_paste_mode = "flip",
    erasing      = 0.50,
    auto_augment = "randaugment",
    close_mosaic = 20,
)

if resuming:
    results = model.train(resume=True)
else:
    results = model.train(**train_kwargs)

RUN_DIR_FINAL = Path(model.trainer.save_dir)
BEST_PT_FINAL = RUN_DIR_FINAL / "weights" / "best.pt"
LAST_PT_FINAL = RUN_DIR_FINAL / "weights" / "last.pt"
print(f"\nEntrenamiento completado.")
print(f"best.pt → {BEST_PT_FINAL}")
print(f"last.pt → {LAST_PT_FINAL}")

## 5. Curvas de entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = RUN_DIR_FINAL / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    fig, axes = plt.subplots(1, 3, figsize=(18, 4))
    for col, lbl in [("train/giou_loss", "train"), ("val/giou_loss", "val")]:
        if col in df.columns: axes[0].plot(df["epoch"], df[col], label=lbl)
    axes[0].set_title("GIoU Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    for col, lbl in [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95")]:
        if col in df.columns: axes[1].plot(df["epoch"], df[col], label=lbl)
    axes[1].set_title("mAP — Bounding Box"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    for col, lbl in [("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall")]:
        if col in df.columns: axes[2].plot(df["epoch"], df[col], label=lbl)
    axes[2].set_title("Precision / Recall"); axes[2].set_xlabel("Epoch"); axes[2].legend()
    plt.suptitle("RT-DETR-L matched a YOLO V4 — curvas", fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()
else:
    print("results.csv no encontrado — ejecuta la celda de entrenamiento primero.")

## 6. Evaluación sobre `val` + log MLflow

In [ ]:
import re
import mlflow
from ultralytics import RTDETR
from datetime import datetime

def sanitize(key):
    return re.sub(r"[^a-zA-Z0-9_\-\. /]", "_", key)

model = RTDETR(str(BEST_PT_FINAL))
m = model.val(
    data=str(DATA_YAML),
    split="val",
    imgsz=IMGSZ,
    conf=0.001,
    iou=0.6,
    plots=True,
    save_json=True,
)
print(f"\nmAP50    (box) : {m.box.map50:.4f}")
print(f"mAP50-95 (box) : {m.box.map:.4f}")
print(f"Precision      : {m.box.mp:.4f}")
print(f"Recall         : {m.box.mr:.4f}")

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment(MLFLOW_EXPERIMENT)
with mlflow.start_run(run_name=f"val_eval_{datetime.now():%Y%m%d_%H%M}"):
    mlflow.log_metrics({
        "val/mAP50_box":    float(m.box.map50),
        "val/mAP50-95_box": float(m.box.map),
        "val/precision":    float(m.box.mp),
        "val/recall":       float(m.box.mr),
    })
    for k, v in m.results_dict.items():
        try:
            mlflow.log_metric(f"val_{sanitize(k)}", float(v))
        except (TypeError, ValueError):
            pass
    mlflow.log_params({
        "matched_to":   "yolo8m_seg_v4",
        "epochs":       EPOCHS,
        "batch":        BATCH,
        "imgsz":        IMGSZ,
        "patience":     PATIENCE,
        "seed":         SEED,
        "optimizer":    "AdamW (RT-DETR default)",
        "lr0":          0.0001,
        "warmup_epochs":3,
        "dataset_ver":  4,
    })
    mlflow.log_artifact(str(BEST_PT_FINAL), artifact_path="model")
    mlflow.log_artifacts(str(RUN_DIR_FINAL), artifact_path="train_artifacts")
    mlflow.set_tags({
        "stage":         "val_evaluation",
        "dataset":       "BlackjackVAI-V4",
        "classes":       "54",
        "backbone":      "rtdetr-l",
        "matched_to":    "yolo8m_seg_v4",
    })
    print("\nLoggeado en MLflow ✓")

## 7. Grid de predicciones sobre val

In [ ]:
import cv2, random
import numpy as np
import matplotlib.pyplot as plt

random.seed(SEED)
pred_model = RTDETR(str(BEST_PT_FINAL))

def pred_grid(img_dir, title, n=9, conf=0.25):
    imgs  = [p for p in img_dir.rglob("*.*") if p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
    samp  = random.sample(imgs, min(n, len(imgs)))
    preds = pred_model.predict(source=[str(p) for p in samp], imgsz=IMGSZ, conf=conf, verbose=False)
    cols  = 3; rows = (len(preds) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(15, 5*rows))
    axes = np.array(axes).flatten()
    for i, res in enumerate(preds):
        ann  = cv2.cvtColor(res.plot(line_width=2), cv2.COLOR_BGR2RGB)
        dets = len(res.boxes) if res.boxes is not None else 0
        axes[i].imshow(ann); axes[i].set_title(f"{Path(res.path).name}  |  {dets} det.", fontsize=8); axes[i].axis("off")
    for j in range(len(preds), len(axes)): axes[j].axis("off")
    plt.suptitle(title, fontsize=13, fontweight="bold")
    plt.tight_layout(); plt.show()

val_dir = DATASET_DIR / ("val" if (DATASET_DIR / "val").exists() else "valid")
pred_grid(val_dir / "images", "Predicciones RT-DETR matched V4 — VAL · conf ≥ 0.25", n=9)

## 8. Exportar modelo final

In [ ]:
import shutil
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M")
dest = MODELS_DIR / f"best_rtdetr_matched_{ts}.pt"
shutil.copy2(str(BEST_PT_FINAL), str(dest))

alias = MODELS_DIR / "best_rtdetr_matched.pt"
if alias.exists() or alias.is_symlink():
    alias.unlink()
try:
    alias.symlink_to(dest.name)
    print(f"Symlink   : {alias} → {dest.name}")
except OSError:
    shutil.copy2(str(dest), str(alias))
    print(f"Copia (Windows sin symlinks): {alias}")

print(f"\nModelo timestamped: {dest}")
print("\nContenido de models/:")
for f in sorted(MODELS_DIR.iterdir()):
    size = f.stat().st_size / 1e6 if f.is_file() else 0
    print(f"  {f.name}  ({size:.1f} MB)")

## 9. Empaquetar artefactos para descarga (solo Colab)

Crea un zip con todo lo necesario para integrar en tu repo local:
- `training_runs/runs/rtdetr_l_matched/` — results.csv, curvas, weights/, args.yaml
- `models/best_rtdetr_matched_<ts>.pt` — pesos finales con timestamp
- `mlflow.db` — DB de Colab con el run logueado por ultralytics

Al ejecutar la celda, en Colab te dispara la descarga automática del zip. En local solo te dice el path.

In [ ]:
import zipfile
from datetime import datetime

ts = datetime.now().strftime("%Y%m%d_%H%M")
zip_path = BASE_DIR / f"rtdetr_matched_export_{ts}.zip"

paths_to_zip = []
# Training run dir
if RUN_DIR_FINAL.exists():
    for p in RUN_DIR_FINAL.rglob("*"):
        if p.is_file():
            paths_to_zip.append((p, p.relative_to(BASE_DIR)))
# Modelo timestamped
for f in MODELS_DIR.glob("best_rtdetr_matched_*.pt"):
    paths_to_zip.append((f, f.relative_to(BASE_DIR)))
# mlflow.db (la de Colab, no la consolidada local)
mlflow_db_local = BASE_DIR / "mlflow.db"
if mlflow_db_local.exists():
    paths_to_zip.append((mlflow_db_local, Path("mlflow_colab.db")))

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
    for src, arc in paths_to_zip:
        zf.write(src, arc)

size_mb = zip_path.stat().st_size / 1e6
print(f"\nZip creado: {zip_path}  ({size_mb:.1f} MB)")
print(f"Contenido: {len(paths_to_zip)} ficheros")

if IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))
    print("\nDescarga iniciada. Cuando termine, llévalo al repo local y avísale a Claude.")
else:
    print(f"\nNo estás en Colab. El zip está en: {zip_path}")